# NutriLens — Food Image Recognition Training

Trains an InceptionV3 model on Food-101 (101 food classes).
**Runtime → Change runtime type → GPU (T4)**

In [ ]:
# ── Step 1: Mount Google Drive (saves model permanently) ─────────────────────
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/NutriLens'
import os
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Model will be saved to: {SAVE_DIR}')

In [ ]:
# ── Step 2: Download & extract Food-101 dataset (~5 GB, ~5-10 min) ──────────
import os
if not os.path.exists('/content/food-101'):
    !wget -q --show-progress http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz
    !tar xzf food-101.tar.gz
    print("Dataset extracted!")
else:
    print("Dataset already exists, skipping download.")
print("Classes:", len(os.listdir("/content/food-101/images")))

In [ ]:
# ── Step 3: Imports ───────────────────────────────────────────────────────────
import os, shutil, stat, collections
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import SGD
from tensorflow.keras import regularizers
from tensorflow.keras.models import load_model

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ── Step 4: Build train/test directory structure ──────────────────────────────
def gen_dir_file_map(path):
    dir_files = defaultdict(list)
    with open(path) as txt:
        for line in txt:
            parts = line.strip().split("/")
            dir_files[parts[0]].append(parts[1] + ".jpg")
    return dir_files

def create_split(src, dst, keep_set):
    """Copy only files listed in keep_set to dst."""    for cls in os.listdir(src):
        src_cls = os.path.join(src, cls)
        dst_cls = os.path.join(dst, cls)
        os.makedirs(dst_cls, exist_ok=True)
        keep = keep_set.get(cls, [])
        for f in keep:
            s = os.path.join(src_cls, f)
            d = os.path.join(dst_cls, f)
            if os.path.exists(s) and not os.path.exists(d):
                shutil.copy2(s, d)

BASE  = "/content/food-101"
TRAIN = BASE + "/train"
TEST  = BASE + "/test"

if not os.path.isdir(TRAIN) or len(os.listdir(TRAIN)) < 101:
    print("Building train/test split...")
    train_map = gen_dir_file_map(BASE + "/meta/train.txt")
    test_map  = gen_dir_file_map(BASE + "/meta/test.txt")
    create_split(BASE + "/images", TRAIN, train_map)
    create_split(BASE + "/images", TEST,  test_map)
    print(f"Train classes: {len(os.listdir(TRAIN))}, Test classes: {len(os.listdir(TEST))}")
else:
    print("Train/test split already exists.")

In [ ]:
# ── Step 5: Data generators ───────────────────────────────────────────────────
IMG_SIZE    = 224   # InceptionV3 standard input size
BATCH_SIZE  = 32    # Larger batch = faster training on T4
N_CLASSES   = 101
TRAIN_SAMPS = 75750
TEST_SAMPS  = 25250

train_gen_obj = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
)
test_gen_obj = ImageDataGenerator(rescale=1./255)

train_gen = train_gen_obj.flow_from_directory(
    TRAIN, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode="categorical"
)
test_gen = test_gen_obj.flow_from_directory(
    TEST, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False
)

print("Train batches:", len(train_gen))
print("Test  batches:", len(test_gen))

In [ ]:
# ── Step 6: Build model ───────────────────────────────────────────────────────
tf.keras.backend.clear_session()

base = InceptionV3(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

# Freeze base; we only train the new head initially
for layer in base.layers:
    layer.trainable = False

x = base.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu", kernel_regularizer=regularizers.l2(0.005))(x)
x = Dropout(0.3)(x)
predictions = Dense(N_CLASSES, activation="softmax", kernel_regularizer=regularizers.l2(0.005))(x)

model = Model(inputs=base.input, outputs=predictions)
model.compile(
    optimizer=SGD(learning_rate=0.0001, momentum=0.9),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print(f"Total params: {model.count_params():,}")
print(f"Trainable:    {sum(tf.size(v).numpy() for v in model.trainable_weights):,}")

In [ ]:
# ── Step 7: Train (30 epochs, ~10-11 hrs on free T4) ─────────────────────────
MODEL_PATH = SAVE_DIR + "/model_trained_101class.hdf5"
LOG_PATH   = SAVE_DIR + "/history_101class.log"

callbacks = [
    ModelCheckpoint(MODEL_PATH, save_best_only=True, monitor="val_accuracy", verbose=1),
    CSVLogger(LOG_PATH, append=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True, verbose=1),
]

history = model.fit(
    train_gen,
    steps_per_epoch=TRAIN_SAMPS // BATCH_SIZE,
    validation_data=test_gen,
    validation_steps=TEST_SAMPS // BATCH_SIZE,
    epochs=30,
    callbacks=callbacks,
    verbose=1,
)

print(f"
✓ Best model saved to: {MODEL_PATH}")

In [ ]:
# ── Step 8: Plot training curves ──────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history["accuracy"],     label="Train",  marker="o", linestyle="--")
ax1.plot(history.history["val_accuracy"], label="Val",    marker="x", linestyle="--")
ax1.set_title("Accuracy — InceptionV3 Food-101")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Accuracy")
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history.history["loss"],     label="Train",  marker="o", linestyle="--")
ax2.plot(history.history["val_loss"], label="Val",    marker="x", linestyle="--")
ax2.set_title("Loss — InceptionV3 Food-101")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss")
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(SAVE_DIR + "/training_curves.png", dpi=120)
plt.show()

In [ ]:
# ── Step 9: Quick sanity test on 6 sample images ──────────────────────────────
import urllib.request
from tensorflow.keras.preprocessing import image as kimage

SAMPLES = {
    "cupcakes":     "https://upload.wikimedia.org/wikipedia/commons/thumb/7/77/Chocolate_cupcakes.jpg/320px-Chocolate_cupcakes.jpg",
    "pizza":        "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a3/Eq_it-na_pizza-margherita_sep2005_sml.jpg/320px-Eq_it-na_pizza-margherita_sep2005_sml.jpg",
    "french_fries": "https://upload.wikimedia.org/wikipedia/commons/thumb/8/83/French_fries_with_salt.jpg/320px-French_fries_with_salt.jpg",
}

foods_sorted = sorted(os.listdir("/content/food-101/images"))
best_model = load_model(MODEL_PATH, compile=False)

fig, axes = plt.subplots(1, len(SAMPLES), figsize=(14, 5))
for ax, (name, url) in zip(axes, SAMPLES.items()):
    path = f"/tmp/{name}.jpg"
    urllib.request.urlretrieve(url, path)
    img = kimage.load_img(path, target_size=(IMG_SIZE, IMG_SIZE))
    arr = np.expand_dims(kimage.img_to_array(img) / 255., axis=0)
    pred = best_model.predict(arr, verbose=0)[0]
    top3 = pred.argsort()[-3:][::-1]
    ax.imshow(img)
    ax.axis("off")
    title = "
".join([f"{foods_sorted[i]}: {pred[i]*100:.1f}%" for i in top3])
    ax.set_title(title, fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ── Step 10: Download model to local machine ──────────────────────────────────
from google.colab import files
print(f"Downloading model from {MODEL_PATH} ...")
files.download(MODEL_PATH)
print("Done! Place model_trained_101class.hdf5 in your NutriLens/ folder.")